# Phase 7.5: exact reconstruction of the type (1,5) record

Phase 7 found a numerical compatible metric with ell² close to sqrt(2/5). Here we reconstruct it exactly from its active logical lifts, certify the polarized abelian surface and relative systole, identify its dual lattice, and prove that it is CM.

In [1]:
from dataclasses import asdict
from pathlib import Path
import json
import sys
import numpy as np

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from gkp_systole import (
    CyclotomicFivePolarization, MetricConvention, SystoleLedgerEntry,
    TYPE_15_CM_ISOGENY_CHANGE, TYPE_15_COMPLEX_STRUCTURE_NUMERATOR,
    TYPE_15_D4_CHANGE, TYPE_15_DUAL_CORE, TYPE_15_EXACT_MODEL,
    TYPE_15_METRIC_CORE, compute_relative_systole,
    cyclotomic_five_moduli_family, reconstruct_equal_distance_metric,
    write_systole_ledger,
)

## Recover the active equal-distance system

At finite precision the classes split by tiny numerical errors, but all 24 nonzero kernel classes approach the same distance. Their nearest lifts have rational coordinates, so equalizing their quadratic lengths is an exact rational linear problem for the ten symmetric Gram entries and ell².

In [2]:
numerical_coordinates = (
    -1.2254741014755715, -1.065892317975433, 0.2705184380885318,
    -1.7218061070489805, -1.210177725742247, 1.1221133976394204,
)
family = cyclotomic_five_moduli_family(CyclotomicFivePolarization(2, -1))
numerical = family.evaluate(numerical_coordinates)
numerical_result = compute_relative_systole(
    family.alternating, numerical.metric,
    metric_convention=MetricConvention.POLARIZATION_SCALED,
)
distances = sorted(float(item.squared_distance) for item in numerical_result.class_results)
lifts = tuple(item.lifts[0] for item in numerical_result.class_results)
reconstructed_shape = reconstruct_equal_distance_metric(lifts)
expected_shape = tuple(tuple(value / 2 for value in row) for row in TYPE_15_METRIC_CORE)
print('number of nonzero classes:', len(distances))
print('distance range:', distances[0], distances[-1])
print('finite-precision spread:', distances[-1] - distances[0])
print('unique rational reconstruction:', reconstructed_shape == expected_shape)
print('G / ell^2 =')
for row in reconstructed_shape:
    print(row)

number of nonzero classes: 24
distance range: 0.632455505731361 0.6324555758180264
finite-precision spread: 7.008666536023611e-08
unique rational reconstruction: True
G / ell^2 =
(Fraction(45, 1), Fraction(-115, 2), Fraction(60, 1), Fraction(-115, 2))
(Fraction(-115, 2), Fraction(75, 1), Fraction(-80, 1), Fraction(75, 1))
(Fraction(60, 1), Fraction(-80, 1), Fraction(110, 1), Fraction(-95, 1))
(Fraction(-115, 2), Fraction(75, 1), Fraction(-95, 1), Fraction(85, 1))


## Exact polarized model and exact ell

Compatibility forces the remaining scale. In integral scaled form, G = G_core/sqrt(10) and J = S/sqrt(10). The identities S² = -10 I and G_core S / 10 = A certify the complex structure and Riemann form. The exact closest-vector calculation gives core ell² = 2, hence physical ell² = 2/sqrt(10) = sqrt(2/5).

In [3]:
model = TYPE_15_EXACT_MODEL
ppav = model.validation_certificate()
exact_result = model.core_relative_systole()
exact_metric = np.asarray(TYPE_15_METRIC_CORE, dtype=float) / np.sqrt(10.0)
numerical_metric = np.asarray(numerical.metric)
print('polarization type:', model.polarization.type)
print('PPAV certified:', ppav.certified)
print('exact core ell^2:', exact_result.squared_systole)
print('physical ell^2:', model.exact_squared_systole, '=', model.squared_systole)
print('shortest classes/lifts:', exact_result.class_multiplicity, exact_result.lift_multiplicity)
print('all class distances:', {item.squared_distance for item in exact_result.class_results})
print('relative metric error from numerical optimizer:', np.linalg.norm(numerical_metric-exact_metric)/np.linalg.norm(exact_metric))

polarization type: (1, 5)
PPAV certified: True
exact core ell^2: 2
physical ell^2: 2/sqrt(10) = sqrt(2/5) = 0.6324555320336759
shortest classes/lifts: 24 24
all class distances: {Fraction(2, 1)}
relative metric error from numerical optimizer: 2.827061830093952e-08


## Why D4 appears

The Euclidean dual Gram matrix is Q/sqrt(10), with Q = 10 G_core^{-1}. An explicit unimodular change of basis sends Q to the D4 root Gram matrix. Thus the 24 nonzero logical classes are represented by the 24 D4 roots, explaining both the exact value and the multiplicity. This identifies a highly symmetric record, but by itself is not a global upper-bound proof for the relative-systole problem.

In [4]:
print('dual core Q =')
for row in TYPE_15_DUAL_CORE:
    print(row)
print('unimodular D4 change =')
for row in TYPE_15_D4_CHANGE:
    print(row)
print('dual is scaled D4:', model.dual_d4_certificate())

dual core Q =
(52, 26, 32, 48)
(26, 14, 15, 22)
(32, 15, 22, 33)
(48, 22, 33, 50)
unimodular D4 change =
(-4, 1, 0, 1)
(5, -1, 0, -2)
(-5, -1, 3, 3)
(5, 0, -2, -2)
dual is scaled D4: True


## CM proof

The integral matrix S defining J satisfies S² = -10 I, so the lattice is stable under the order Z[sqrt(-10)] of discriminant -40. A rational change of basis of determinant 47 splits S into two identical 2 by 2 companion blocks. Therefore the surface is isogenous to the square of the elliptic curve with period i sqrt(10). Its rational endomorphism algebra contains M₂(Q(sqrt(-10))) and hence a commutative semisimple algebra of dimension four: the surface is CM.

In [5]:
cm = model.cm_certificate()
print('CM certified:', cm.is_cm)
print('CM field:', cm.field)
print('order discriminant:', cm.order_discriminant)
print('elliptic factor period:', cm.elliptic_period)
print('rational commutant dimension:', cm.commutant_dimension)
print('isogeny lattice index:', cm.rational_isogeny_degree)
print('integral S =')
for row in TYPE_15_COMPLEX_STRUCTURE_NUMERATOR:
    print(row)
print('isogeny change =')
for row in TYPE_15_CM_ISOGENY_CHANGE:
    print(row)
print('block form =')
for row in cm.block_matrix:
    print(row)

CM certified: True
CM field: Q(sqrt(-10))
order discriminant: -40
elliptic factor period: i*sqrt(10)
rational commutant dimension: 8
isogeny lattice index: 47
integral S =
(68, -88, 96, -90)
(35, -44, 42, -42)
(41, -53, 68, -61)
(61, -80, 104, -92)
isogeny change =
(1, 68, 0, -88)
(0, 35, 1, -44)
(0, 41, 0, -53)
(0, 61, 0, -80)
block form =
(0, -10, 0, 0)
(1, 0, 0, 0)
(0, 0, 0, -10)
(0, 0, 1, 0)


## Update the project ledger

The numerical control is retained for provenance, and a new exact CM entry records its reconstruction. Global optimality is still not claimed.

In [6]:
json_path = ROOT / 'data' / 'phase7_result_ledger.json'
csv_path = ROOT / 'data' / 'phase7_result_ledger.csv'
existing = json.loads(json_path.read_text())
entries = [SystoleLedgerEntry(**record) for record in existing if record['candidate_id'] != 'type_1_5_exact_cm_reconstruction']
entries.append(SystoleLedgerEntry(
    candidate_id='type_1_5_exact_cm_reconstruction', phase=7, dimension_g=2,
    polarization_type='(1, 5)', family='exact D4-dual CM surface',
    cm_data='isogenous (index 47) to E_(i*sqrt(10))^2; order discriminant -40',
    ell_squared_decimal=f'{model.squared_systole:.16g}',
    ell_squared_exact=model.exact_squared_systole,
    class_multiplicity=exact_result.class_multiplicity,
    lift_multiplicity=exact_result.lift_multiplicity,
    metric_convention='polarization_scaled',
    arithmetic_status='exact PPAV, CVP, D4, and CM certificates',
    search_status='best observed Phase-7 candidate; global optimality open',
    search_scope='exact reconstruction of the full-dimensional numerical record',
    notes='all 24 nonzero kernel classes have equal distance',
))
write_systole_ledger(entries, json_path=json_path, csv_path=csv_path)
print('updated ledger entries:', len(entries))
print('wrote:', json_path)
print('wrote:', csv_path)

updated ledger entries: 12
wrote: data/phase7_result_ledger.json
wrote: data/phase7_result_ledger.csv
